# Comparación de Presupuestos de Construcción - Test de Viabilidad

Este notebook prueba las capacidades de matching semántico para comparación de presupuestos.

## Instrucciones:
1. Subir este notebook a Google Colab
2. Agregar API key a Secrets de Colab (icono de llave) con nombre 'GOOGLE_API_KEY'
3. Subir archivos de presupuesto (CSV, Excel, PDF)
4. Ejecutar todas las celdas

## Qué prueba:
- Matching semántico entre descripciones diferentes
- Generación de tabla de comparación
- Exportación a CSV/Excel

In [ ]:
# Instalar dependencias
!pip install -q google-generativeai pandas openpyxl

In [ ]:
# Importar librerías
import google.generativeai as genai
from google.colab import userdata, files
import pandas as pd
import json
import io

# Configurar API
try:
    API_KEY = userdata.get('GOOGLE_API_KEY')
except:
    API_KEY = input('Ingresa tu API key de Gemini: ')

genai.configure(api_key=API_KEY)
print('✓ Gemini configurado correctamente')

In [ ]:
# Subir archivos de presupuesto
print('Sube tus archivos de presupuesto (CSV, Excel, o PDF):')
print('1. Archivo base (presupuesto de referencia)')
print('2. Archivo(s) de comparación (cotizaciones de proveedores)')
print()
uploaded = files.upload()

file_names = list(uploaded.keys())
print(f'\n✓ Subidos {len(file_names)} archivos: {file_names}')

In [ ]:
# Cargar datos de archivos (con soporte para PDF)
def load_file_data(filename):
    """Carga archivos CSV, Excel, o PDF."""
    if filename.endswith('.csv'):
        with open(filename, 'r', encoding='utf-8') as f:
            return ('text', f.read())
    elif filename.endswith(('.xlsx', '.xls')):
        df = pd.read_excel(filename)
        return ('text', df.to_csv(index=False))
    elif filename.endswith('.pdf'):
        # Para PDFs, usar API de upload de Gemini
        print(f'  Subiendo PDF a Gemini API: {filename}...')
        uploaded_file = genai.upload_file(filename)
        return ('file', uploaded_file)
    else:
        raise ValueError(f'Tipo de archivo no soportado: {filename}')

# Cargar todos los archivos
file_data = {}
pdf_files = []

for fname in file_names:
    ftype, fdata = load_file_data(fname)
    file_data[fname] = (ftype, fdata)
    if ftype == 'file':
        pdf_files.append(fdata)
    print(f'✓ Cargado: {fname}')

print(f'\nTotal archivos cargados: {len(file_data)}')

In [ ]:
# Especificar archivo base
print('Archivos disponibles:')
for i, fname in enumerate(file_names):
    print(f'{i+1}. {fname}')

base_idx = int(input('\nIngresa número del archivo BASE: ')) - 1
base_file = file_names[base_idx]
comparison_files = [f for f in file_names if f != base_file]

print(f'\n✓ Archivo base: {base_file}')
print(f'✓ Archivos de comparación: {comparison_files}')

## Test 1: Matching Semántico

Este test pide a Gemini que identifique ítems coincidentes usando comprensión semántica.

In [ ]:
# Crear modelo con chat (para mantener contexto)
model = genai.GenerativeModel('gemini-1.5-flash')
chat = model.start_chat(history=[])

print('✓ Chat iniciado (mantiene contexto entre mensajes)')

In [ ]:
# Construir prompt con los datos
prompt_parts = []
prompt_parts.append("""
Necesito comparar ítems de presupuestos de construcción entre múltiples archivos.
Los ítems están descritos de forma diferente pero muchos significan lo mismo.
Usa matching SEMÁNTICO para identificar ítems equivalentes.

""")

# Agregar archivo base
ftype, fdata = file_data[base_file]
if ftype == 'text':
    prompt_parts.append(f"ARCHIVO BASE ({base_file}):\n{fdata}\n\n")
else:
    prompt_parts.append(f"ARCHIVO BASE ({base_file}): [Ver PDF adjunto]\n\n")

# Agregar archivos de comparación
for i, comp_file in enumerate(comparison_files, 1):
    ftype, fdata = file_data[comp_file]
    if ftype == 'text':
        prompt_parts.append(f"ARCHIVO COMPARACIÓN {i} ({comp_file}):\n{fdata}\n\n")
    else:
        prompt_parts.append(f"ARCHIVO COMPARACIÓN {i} ({comp_file}): [Ver PDF adjunto]\n\n")

prompt_parts.append("""
TAREA:
Para cada ítem CONCRETO del archivo BASE (ítems con precios, NO encabezados de sección),
identifica ítems coincidentes de los archivos de comparación.

IMPORTANTE:
- Usa matching SEMÁNTICO (no matching exacto de texto)
- Considera sinónimos, abreviaciones, variaciones de orden de palabras
- "LOCALIZACIÓN Y REPLANTEO POR METRO CUADRADO..." debe coincidir con "LOCALIZACIÓN Y REPLANTEO"
- "DEMOLICIÓN DE MUROS" debe coincidir con "DEMOLER MURO"
- "EXCAVACIÓN MANUAL EN TIERRA" debe coincidir con "EXCAVACIÓN MANUAL EN SUELO"

Para CADA ítem base con precio, proporciona:
1. Número y descripción del ítem base (abreviado a 50 caracteres)
2. Coincidencias de cada archivo de comparación (número + descripción, o "NO ENCONTRADO")
3. Nivel de confianza (ALTO/MEDIO/BAJO) para cada coincidencia
4. Razonamiento breve

Sé exhaustivo y sistemático.
""")

# Agregar PDFs si existen
full_prompt = []
for part in prompt_parts:
    full_prompt.append(part)
for pdf in pdf_files:
    full_prompt.append(pdf)

print('Enviando solicitud a Gemini...')
print('(Esto puede tomar 30-60 segundos...)\n')

In [ ]:
# Enviar a Gemini (PRIMERA MENSAJE - matching semántico)
response1 = chat.send_message(full_prompt)

print('='*60)
print('RESULTADOS DE MATCHING SEMÁNTICO')
print('='*60)
print()
print(response1.text)
print()
print('='*60)

## Test 2: Generar Tabla de Comparación

Ahora pedimos a Gemini que formatee los resultados como tabla estructurada.

**Nota:** Usa el contexto del mensaje anterior (gracias al chat).

In [ ]:
# Generar tabla de comparación (SEGUNDO MENSAJE - usa contexto)
table_prompt = f"""
Basándote en tu análisis anterior, crea una tabla de comparación en formato markdown.

Columnas:
- Ítem Base (abreviado)
- Unidad Base
- Precio Base
"""

for i, comp_file in enumerate(comparison_files, 1):
    table_prompt += f"""
- Ítem {comp_file}
- Unidad {comp_file}
- Precio {comp_file}
- Coincide (✓/✗)
"""

table_prompt += """

Reglas:
- Incluye solo ítems concretos con precios (no encabezados de sección)
- Usa descripciones abreviadas (máximo 40 caracteres)
- Muestra precios reales de los datos
- Usa ✓ para coincidencias, ✗ para no encontrado
- Al final, muestra estadísticas de coincidencias

Sé preciso y completo.
"""

response2 = chat.send_message(table_prompt)

print('='*60)
print('TABLA DE COMPARACIÓN')
print('='*60)
print()
print(response2.text)
print()
print('='*60)

## Test 3: Exportar como CSV

Obtener los datos de comparación en formato CSV para importar a Excel.

In [ ]:
# Generar formato CSV (TERCER MENSAJE - aún tiene contexto)
csv_headers = ['Item_Base', 'Unidad_Base', 'Precio_Base']
for i, comp_file in enumerate(comparison_files, 1):
    nombre_corto = f'Archivo{i}'
    csv_headers.extend([
        f'Item_{nombre_corto}',
        f'Unidad_{nombre_corto}',
        f'Precio_{nombre_corto}',
        f'Coincide_{nombre_corto}'
    ])

csv_prompt = f"""
Ahora genera los datos de comparación en formato CSV (valores separados por comas).

Encabezados CSV:
{','.join(csv_headers)}

Reglas:
- Descripciones cortas de ítems (máximo 40 caracteres)
- Solo precios numéricos (elimina símbolos $ y comas)
- SI/NO para columnas de coincidencia
- Una fila por ítem base
- Encierra campos de texto entre comillas si contienen comas

Proporciona SOLO los datos CSV (sin explicaciones, sin bloques de código markdown).
Inicia directamente con la fila de encabezados.
"""

response3 = chat.send_message(csv_prompt)

print('='*60)
print('SALIDA CSV')
print('='*60)
print()
print(response3.text)
print()
print('='*60)

# Guardar a archivo
with open('comparacion_resultados.csv', 'w', encoding='utf-8') as f:
    # Eliminar bloques de código markdown si existen
    csv_data = response3.text.replace('```csv', '').replace('```', '').strip()
    f.write(csv_data)

print('\n✓ Guardado en comparacion_resultados.csv')
print('\nDescargar el archivo:')
files.download('comparacion_resultados.csv')

## Evaluación

Revisa los resultados anteriores para evaluar:

### Verificación de Precisión:
- ¿Identificó Gemini correctamente las coincidencias semánticas?
- ¿Se emparejaron correctamente los sinónimos? (ej. DEMOLICIÓN ↔ DEMOLER)
- ¿Se manejaron las variaciones de orden de palabras?
- ¿Se expandieron correctamente las abreviaciones?

### Rendimiento Esperado:
- **Alta precisión (85-95%)**: La mayoría de coincidencias correctas
- **Precisión media (70-85%)**: Algunos errores, necesita revisión
- **Baja precisión (<70%)**: El enfoque necesita refinamiento

### Próximos Pasos:
Si la precisión es buena (>80%), ¡proceder con la implementación completa!

## Conclusión de Viabilidad:

✅ **Si las coincidencias se ven bien**: El enfoque de matching semántico FUNCIONA para tu caso de uso

✅ **Siguiente**: Implementar solución completa con:
- Extracción estructurada de tablas (pdfplumber + pandas)
- Matching híbrido (embeddings + LLM)
- Exportación a Excel con formato
- Flujo de trabajo de revisión manual

📊 **Implementación**: 1-2 semanas para solución lista para producción